## В этой работе собирается система, которая читает загруженный документ и отвечает на вопросы по его содержанию. Архитектура основана на подходе Retrieval-Augmented Generation (RAG): сначала система находит в документе фрагменты, релевантные вопросу, а затем языковая модель формулирует ответ, опираясь только на эти фрагменты.

# **Модель 1 — Эмбеддер: intfloat/multilingual-e5-small**

Ссылка: huggingface.co/intfloat/multilingual-e5-small

Модель семейства E5, предназначенная для преобразования текста в векторные представления (эмбеддинги). Поддерживает более 100 языков, включая русский. Имеет 12 слоёв, размерность выходного вектора — 384, максимальная длина последовательности — 512 токенов, около 118 млн параметров. Модель компактная (примерно 0.44 ГБ в fp32), поэтому быстро работает на GPU T4 и не создаёт конкуренции за память с языковыми моделями.

E5-модели обучены с префиксами: для документов используется passage:, для запросов — query:. Эти префиксы не опциональны — без них качество поиска заметно снижается.

Роль в системе: превращает каждый фрагмент документа в вектор при загрузке, а затем превращает вопрос (после переписывания) в вектор, чтобы найти ближайшие фрагменты по косинусной близости.

# Модель 2 — Answerer: Qwen/Qwen2.5-1.5B-Instruct
Ссылка: huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

Та же серия Qwen2.5, но с 1.5 млрд параметров. По бенчмаркам значительно превосходит 0.5B-версию при сохранении относительно небольших вычислительных требований. Модель занимает около 1.64 ГБ в fp16, помещается на T4 без проблем. Контекстное окно — 33K токенов, что с запасом покрывает объём типичного RAG-запроса (несколько фрагментов текста плюс история диалога).

Роль в системе: пополучает вопрос пользователя вместе с найденными фрагментами документа и формулирует ответ. Системная инструкция ограничивает модель: отвечать только по предоставленному тексту, не выдумывать факты, при отсутствии ответа — честно сообщить об этом.


# Принцип работы:


```
 Вопрос пользователя
        ↓
    multilingual-e5-base  →  кодирует вопрос, находит top-k фрагментов
        ↓
    Qwen2.5-1.5B-Instruct  →  формулирует ответ по найденным фрагментам
```

Две модели работают последовательно, каждая решает свою задачу. Общая задержка на один вопрос на GPU T4 составляет примерно 2–4 секунды (большая часть времени уходит на генерацию ответа).


# Установка зависимостей

In [1]:
!pip -q install "transformers>=4.42.0" accelerate sentence-transformers pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.6 MB/s eta 0:00:00


# Установка зависимостей

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Эмбеддер для поиска по документу
EMB_MODEL = "intfloat/multilingual-e5-base"
embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)

# Языковая модель для генерации ответа
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)

print("Модели загружены. Устройство:", DEVICE)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Модели загружены. Устройство: cuda


# Класс RAG-ассистента с Query Rewriter

In [3]:
import numpy as np
from docx import Document
from pypdf import PdfReader


class RAGAssistant:
    def __init__(self, embedder, tok, llm, chunk_size=500, overlap=100, top_k=3):
        self.embedder = embedder
        self.tok = tok
        self.llm = llm
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_k = top_k
        self.chunks = []
        self.embeddings = None
        self.source_name = None

    # Загрузка текста

    def _split_text(self, text):
        # Режет текст на перекрывающиеся фрагменты
        text = text.strip()
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start = end - self.overlap
        return chunks

    def load_text(self, text, source_name="текст"):
        self.chunks = self._split_text(text)
        # Префикс passage: обязателен для моделей E5
        prepared = [f"passage: {c}" for c in self.chunks]
        self.embeddings = self.embedder.encode(
            prepared, normalize_embeddings=True, show_progress_bar=True
        )
        self.source_name = source_name
        print(f"Загружено {len(self.chunks)} фрагментов из «{source_name}».")

    def _read_docx(self, path):
        # Извлекает параграфы и таблицы из DOCX
        doc = Document(path)
        parts = []
        for p in doc.paragraphs:
            if p.text.strip():
                parts.append(p.text)
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(" | ".join(cells))
        return "\n".join(parts)

    def _read_pdf(self, path):
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    def _read_txt(self, path):
        # Пробует разные кодировки
        for enc in ("utf-8", "cp1251", "latin-1"):
            try:
                with open(path, "r", encoding=enc) as f:
                    return f.read()
            except UnicodeDecodeError:
                continue
        raise ValueError("Не удалось определить кодировку файла.")

    def load_file(self, path):
        lower = path.lower()
        if lower.endswith(".pdf"):
            text = self._read_pdf(path)
        elif lower.endswith(".docx"):
            text = self._read_docx(path)
        elif lower.endswith(".doc"):
            raise ValueError(
                "Старый формат .doc не поддерживается. "
                "Сохраните файл как .docx или .txt."
            )
        else:
            text = self._read_txt(path)
        self.load_text(text, source_name=path.split("/")[-1])

    # Поиск и ответ

    def _retrieve(self, query, top_k=None):
        # Ищет top-k фрагментов по косинусной близости
        top_k = top_k or self.top_k
        # Префикс query: обязателен для моделей E5
        q_emb = self.embedder.encode(
            f"query: {query}", normalize_embeddings=True
        )
        scores = self.embeddings @ q_emb
        idx = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in idx]

    def ask(self, question, history=None, verbose=False):
        if not self.chunks:
            return "Сначала загрузите текст."

        # Находим релевантные фрагменты
        retrieved = self._retrieve(question)
        context = "\n\n---\n\n".join(chunk for chunk, _ in retrieved)

        if verbose:
            print("[Поиск] Найденные фрагменты:")
            for chunk, score in retrieved:
                print(f"  [{score:.3f}] {chunk[:120]}...")
            print()

        # Инструкция для модели
        system = (
            "Ты — ассистент, который отвечает на вопросы строго по "
            "предоставленному тексту. Правила:\n"
            "1. Используй только информацию из контекста.\n"
            "2. Если в контексте сказано, что что-то не выяснено, "
            "осталось нерешённым, требует изучения или не получило "
            "объяснения — ответь ровно одной фразой, например: "
            "«В тексте сказано, что природа этого явления осталась "
            "невыясненной». Не добавляй объяснений, не предлагай версий.\n"
            "3. Не добавляй факты, которых нет в контексте.\n"
            "4. Если в контексте есть слова из вопроса, но они относятся "
            "к другому году, объекту или событию — это не ответ. Скажи: "
            "«В загруженном тексте этого нет».\n"
            "5. Отвечай кратко, по-русски.\n\n"
            "ПРИМЕР правильного ответа на вопрос-сравнение. Обрати "
            "внимание: в контексте есть лишнее число, которое НЕ "
            "относится к сравнению, и оно НЕ используется в ответе.\n\n"
            "Вопрос: Сравните глубины в 1950 и 1980 годах. На сколько "
            "они различаются?\n"
            "Контекст: «В 1950 году глубина составляла 20 метров. "
            "В 1980 году — 25 метров. На глубине 15 метров "
            "зафиксирована аномалия температур».\n"
            "Ответ: В 1950 году — 20 метров. В 1980 году — 25 метров. "
            "Разница: 25 − 20 = 5 метров. Число 15 относится к аномалии "
            "температур и в сравнении глубин не используется.\n\n"
            "Правило для вопросов-сравнений: используй ТОЛЬКО те числа, "
            "которые прямо относятся к сравниваемым объектам. Игнорируй "
            "все остальные числа в контексте, даже если они выглядят "
            "похоже."
        )
        user = f"Контекст:\n{context}\n\nВопрос: {question}"

        # Собираем сообщения в формате чата
        messages = [{"role": "system", "content": system}]
        if history:
            for u, a in history:
                messages.append({"role": "user", "content": u})
                messages.append({"role": "assistant", "content": a})
        messages.append({"role": "user", "content": user})

        text = self.tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tok(text, return_tensors="pt").to(self.llm.device)

        # Генерация ответа
        with torch.no_grad():
            out = self.llm.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                pad_token_id=self.tok.eos_token_id,
            )
        # Отрезаем промпт, оставляем только ответ модели
        answer = self.tok.decode(
            out[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()
        return answer


rag = RAGAssistant(embedder=embedder, tok=tok, llm=llm, top_k=3)
print("RAG-ассистент готов.")

RAG-ассистент готов.


# Загрузка файла

In [4]:
from google.colab import files

uploaded = files.upload()
path = list(uploaded.keys())[0]
rag.load_file(path)

Saving test_text.docx to test_text.docx


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Загружено 8 фрагментов из «test_text.docx».


# Диалог

In [5]:
history = []
print("Ассистент готов. Задавайте вопросы. Для выхода введите: выход\n")

while True:
    try:
        q = input("Вы: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nЗавершено.")
        break

    if not q:
        continue
    if q.lower() in {"выход", "exit", "quit", "q"}:
        print("Завершено.")
        break

    answer = rag.ask(q, history=history, verbose=True)
    print(f"\nАссистент: {answer}\n")
    history.append((q, answer))

Ассистент готов. Задавайте вопросы. Для выхода введите: выход

Вы: Какова максимальная глубина озера по данным экспедиции 1987 года?
[Поиск] Найденные фрагменты:
  [0.867] ну 28 метров. В 1959 году 
экспедиция Московского университета под руководством Веры Шапошниковой 
уточнила данные: глуб...
  [0.852] одного 
зеркала составляет 12,4 гектара, максимальная глубина — 33,4 метра. 
Берега озера имеют форму правильного овала,...
  [0.818] мках экспедиции и требуют дополнительного 
изучения.
Часть 6. Выводы
Экспедиция подтвердила тектоническую гипотезу проис...


Ассистент: 33,4 метра

Вы: Сравните максимальные глубины, зафиксированные в 1912 и 1987 годах. На сколько метров они различаются?
[Поиск] Найденные фрагменты:
  [0.808] ну 28 метров. В 1959 году 
экспедиция Московского университета под руководством Веры Шапошниковой 
уточнила данные: глуб...
  [0.799] одного 
зеркала составляет 12,4 гектара, максимальная глубина — 33,4 метра. 
Берега озера имеют форму правильного овала,...
  [0.79